# 8. La submission

El modelo final se entrena con todo train y predice el test. Lo hace `python correr.py submission`, que se niega a generar el archivo si la configuración no es la misma que se validó.

```
python correr.py submission --validacion <id de la validación>
python correr.py anotar <id de la submission> --puntaje 0.95
```

El archivo tiene las columnas `ID` y `condition`, donde `ID` es el número de fila del test empezando en 1.

In [1]:
import sys
import warnings

sys.path.insert(0, "..")  # Los módulos están en la raíz del proyecto
warnings.filterwarnings("ignore")
%matplotlib inline

import pandas as pd

import datos
import modelo

corridas = modelo.leer()
puntajes = {r["submission"]: r["leaderboard_publico"] for r in corridas if r["tipo"] == "leaderboard"}
ultima = [r for r in corridas if r["tipo"] == "submission"][-1]
archivo = pd.read_csv(datos.RAIZ / ultima["archivo"])
print(ultima["id"], "validada con", ultima["run_validacion"])
print(len(archivo), "filas y las columnas", list(archivo.columns))
print(f"Predice {ultima['fraccion_predicha_usado']:.1%} de usados (en train hay {ultima['fraccion_usado_en_train']:.1%})")
archivo.head()

20261110-002606-submission-lightgbm validada con 20261110-001656-validacion-lightgbm
30000 filas y las columnas ['ID', 'condition']
Predice 47.1% de usados (en train hay 46.3%)


,ID,condition
0,1,used
1,2,new
2,3,used
3,4,used
4,5,new


## Contra la mejor submission anterior

El promedio de cuatro modelos es la mejor anotada hasta ahora. Se cuenta en cuántos items cambia la predicción.

In [2]:
anterior = pd.read_csv(datos.RAIZ / "outputs/submissions/20261102-120209-ensamble.csv")
distintas = (anterior["condition"] != archivo["condition"]).sum()
print(f"Cambian {distintas} de {len(archivo)} predicciones ({100 * distintas / len(archivo):.2f}%)")

Cambian 175 de 30000 predicciones (0.58%)


## Todas las submissions

Para las dos finales de Kaggle conviene elegir por validación, no por el leaderboard público, porque ese puntaje mira solo una parte del test y perseguirlo es sobreajustar.

In [3]:
pd.DataFrame([
    {"submission": r["id"], "validación": round(r["accuracy_validacion"], 5),
     "leaderboard": puntajes.get(r["id"], "sin cargar")}
    for r in corridas if r["tipo"] == "submission"
])

,submission,validación,leaderboard
0,20261010-195207-submission-lightgbm,0.94288,0.94352
1,20261011-151252-submission-ensamble,0.94349,sin cargar
2,20261011-180636-submission-ensamble,0.94374,0.9439
3,20261025-201009-submission-lightgbm,0.94743,sin cargar
4,20261025-212849-submission-ensamble,0.94824,0.94747
5,20261102-120209-submission-ensamble,0.94739,0.94761
6,20261102-201303-submission-ensamble,0.94739,sin cargar
7,20261104-132134-submission-lightgbm,0.94700,sin cargar
8,20261110-002606-submission-lightgbm,0.94700,sin cargar
